# Reto: Deserción de Empleados
### Ingeniería de Características

In [ ]:
# Paso 1: Importar librerías
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

In [ ]:
# Paso 2: Leer el archivo CSV
EmpleadosAttrition = pd.read_csv('empleadosRETO.csv')
EmpleadosAttrition.head()

In [ ]:
# Paso 3: Eliminar columnas irrelevantes
EmpleadosAttrition.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], inplace=True)
EmpleadosAttrition.head()

In [ ]:
# Paso 5: Crear columna Year a partir de HiringDate (extraer el año como entero)
EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].str.split('/').str[2].str[:4].astype(int)
EmpleadosAttrition['Year'].head()

In [ ]:
# Paso 6 (Paso 7 en instrucciones): Crear columna YearsAtCompany (años en la empresa hasta 2018)
EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']
EmpleadosAttrition['YearsAtCompany'].head()

In [ ]:
# Paso 8: Renombrar DistanceFromHome a DistanceFromHome_km
EmpleadosAttrition.rename(columns={'DistanceFromHome': 'DistanceFromHome_km'}, inplace=True)

# Paso 9: Crear nueva columna DistanceFromHome sin las letras 'km' y como entero
EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.replace(' km', '').astype(int)

EmpleadosAttrition[['DistanceFromHome_km', 'DistanceFromHome']].head()

In [ ]:
# Paso 10: Borrar columnas Year, HiringDate y DistanceFromHome_km
EmpleadosAttrition.drop(columns=['Year', 'HiringDate', 'DistanceFromHome_km'], inplace=True)
EmpleadosAttrition.head()

In [ ]:
# Paso 11: Sueldo promedio por departamento (solo informativo)
SueldoPromedioDepto = EmpleadosAttrition.groupby('Department')[['MonthlyIncome']].mean()
SueldoPromedio = SueldoPromedioDepto
print(SueldoPromedio)

In [ ]:
# Paso 12: Escalar MonthlyIncome entre 0 y 1 (Min-Max)
scaler = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = scaler.fit_transform(EmpleadosAttrition[['MonthlyIncome']])
EmpleadosAttrition['MonthlyIncome'].describe()

In [ ]:
# Paso 13: Convertir variables categóricas a numéricas con LabelEncoder
cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'Attrition']
le = LabelEncoder()
for col in cat_cols:
    EmpleadosAttrition[col] = le.fit_transform(EmpleadosAttrition[col])

EmpleadosAttrition.head()

In [ ]:
# Paso 14: Correlación lineal de cada variable con Attrition
correlaciones = EmpleadosAttrition.corr()['Attrition'].abs()
print(correlaciones.sort_values(ascending=False))

In [ ]:
# Paso 15: Seleccionar variables con correlación >= 0.1 (incluyendo Attrition)
keep_cols = correlaciones[correlaciones >= 0.1].index.tolist()
print('Variables seleccionadas:', keep_cols)
print('Variables eliminadas:', [c for c in EmpleadosAttrition.columns if c not in keep_cols])

EmpleadosAttritionFinal = EmpleadosAttrition[keep_cols].copy()
EmpleadosAttritionFinal.head()

In [ ]:
# Paso 16: PCA sobre EmpleadosAttritionFinal
pca = PCA()
EmpleadosAttritionPCA = pca.fit_transform(EmpleadosAttritionFinal)

# Ver varianza explicada acumulada
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)
print('Varianza explicada acumulada por componente:')
for i, v in enumerate(varianza_acumulada):
    print(f'  C{i}: {v:.4f}')

In [ ]:
# Paso 17: Agregar el mínimo número de Componentes Principales que expliquen el 80% de la varianza
# Con C0 = 63.5% y C1 = 87.7%, se necesitan 2 componentes (C0 y C1)
EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(C0=EmpleadosAttritionPCA[:,0])
EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(C1=EmpleadosAttritionPCA[:,1])

print(f'Varianza explicada con C0: {pca.explained_variance_ratio_[0]:.4f}')
print(f'Varianza explicada con C0+C1: {varianza_acumulada[1]:.4f}')
print(f'\nShape final: {EmpleadosAttritionFinal.shape}')
EmpleadosAttritionFinal.head()

In [ ]:
# Paso 18: Guardar el dataset final en CSV
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)
print('Archivo guardado: EmpleadosAttritionFinal.csv')
print('Columnas finales:', list(EmpleadosAttritionFinal.columns))